# Local LLM Model Evaluation Demo

This notebook asks each local LLM model to write the same Python program many times. Then, it runs each generated program in a clean Python Docker container and counts how often the program works.

This notebook is written for the 2026 UNC Computer Science XL AI Workshop.

The notebook is organized as the experiment story: choose settings, generate programs, run tests, compare results, and inspect failures. The reusable code for generation, Docker execution, evaluation, and display lives in `helper_functions.py` so the notebook can stay focused on the evaluation workflow.

Be sure to run `uv sync` in this project's directory to install required Python library dependencies.

## Settings

This cell is the control panel for the experiment. The list of `MODELS` are those being evaluated. If you were able to run a larger model than `2b`, feel free to add it to the list or replace `2b`.

The model output files are saved to the `generated-programs/` directory, which is ignored by Git.

Each generation uses its attempt counter as the Ollama seed. Seeding controls the randomness of the LLM, so regenerating seed `001` should result in the same output each time, if the prompt, model, and generation settings stay the same.

`SYSTEM_PROMPT` and `PROMPT` are separated so we can discuss the difference between role-level instructions and task-level instructions.

In [ ]:
from pathlib import Path

# Experimental Setup:

MODELS = [
    "qwen3.5:0.8b",
    # "qwen3.5:??", # TODO - Enable largest model your machine can run. 
]

SYSTEM_PROMPT = (
    "You are a Python programmer who writes simple, well structured programs. "
    "When asked to write a program, respond with a complete Python program. Code only. "
    "Do not respond with markdown. Do not respond with ```python code fences." 
)

PROMPT = (
    "Write a Python program that takes two ints as command-line arguments and prints their sum.\n"
)

N_SAMPLES = 10

TEST_CASES = [
    (["1", "2"], "3"),
    (["10", "25"], "35"),
    (["-4", "9"], "5"),
    (["0", "0"], "0"),
]

# Ollama Generation Parameters
THINK = False
MAX_GENERATED_TOKENS = 1024
TEMPERATURE = 0.2
OUTPUT_DIR = Path("generated-programs")

# Docker Image to Evaluate the Generated Code With
DOCKER_IMAGE = "python:3.12"

## Generate Programs

The next code cell imports the helper functions used by the rest of the notebook. Those helpers are in `helper_functions.py`; the notebook passes in the settings from above so the experiment remains easy to modify here.

The generation cell clears `generated-programs/`, asks each model to generate `N_SAMPLES` programs, and saves each response exactly as the model wrote it. If the model adds Markdown fences or explanation text, the generated `.py` file should fail. That is part of what we are measuring.

In [ ]:
from helper_functions import (
    evaluate_model,
    first_failure,
    generate_programs,
    reset_output_dir,
    show_summary_table,
    summarize_results,
)

In [ ]:
# This cell will take some time to complete
reset_output_dir(OUTPUT_DIR)

for model_name in MODELS:
    generate_programs(
        model_name=model_name,
        count=N_SAMPLES,
        output_dir=OUTPUT_DIR,
        prompt=PROMPT,
        system_prompt=SYSTEM_PROMPT,
        think=THINK,
        max_generated_tokens=MAX_GENERATED_TOKENS,
        temperature=TEMPERATURE,
    )

## Evaluate Programs

The evaluation helpers also live in `helper_functions.py`. Evaluation runs each Python program in an ephemeral Docker container. A generated program succeeds only if it passes every test case with no error and exactly the expected output.

The next code cell evaluates every saved program for every model and stores both the full per-file results and a compact summary.

In [ ]:
all_results = {}
summary = {}

for model_name in MODELS:
    rows = evaluate_model(
        model_name=model_name,
        output_dir=OUTPUT_DIR,
        test_cases=TEST_CASES,
        docker_image=DOCKER_IMAGE,
    )
    all_results[model_name] = rows
    summary[model_name] = summarize_results(rows)

summary

## Compare Models

The summary table shows the main result of the experiment: how often each model produced a working program under the same prompt, test cases, and generation settings.

In [ ]:
show_summary_table(summary)

## Look At A Few Failures

This cell prints a few failing files and their first failing test. Open the `.py` file to see what the model wrote, then compare the failure with the prompt and system prompt above.

In [ ]:
for model_name, rows in all_results.items():
    failures = [row for row in rows if not row["passed"]]
    print(f"\n{model_name}: showing {min(5, len(failures))} of {len(failures)} failures")

    for row in failures[:5]:
        failure = first_failure(row)
        print(row["script"])
        print("  args:", failure["args"])
        print("  expected:", repr(failure["expected"]))
        print("  actual:", repr(failure["actual"]))
        print("  stderr:", failure["stderr"].splitlines()[-1:] or "")